# Sentinel Vision — Demo Notebook (Phase 4)

Full-stack video surveillance analytics:
detection → tracking → zones → gates → dwell → loitering → abandoned objects
→ calibration → interactions → evidence capture → **vehicle ANPR**

---
## Setup

Clone the Phase 4 branch and install dependencies.

In [ ]:
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/basil-sami/sentinel-vision"
BRANCH = "phase-4-vehicle-intelligence"
REPO_DIR = "sentinel-vision"

if not Path(REPO_DIR).exists():
    !git clone --branch {BRANCH} --single-branch {REPO_URL}
else:
    %cd {REPO_DIR}
    !git checkout {BRANCH} && git pull

%cd {REPO_DIR}
!pip install -q -r requirements.txt

# Install EasyOCR for ANPR (lightweight, works on CPU)
!pip install -q easyocr

# autoreload is optional; safely skip on Python 3.12+ where `imp` is removed
try:
    %load_ext autoreload
    %autoreload 2
except Exception:
    pass
print("\nReady.")

## Download or Upload Video

Tries the default test video via yt-dlp. Falls back to manual upload.

In [ ]:
DEFAULT_VIDEO = "The CCTV People Demo 2.mp4"
DEFAULT_URL = "https://www.youtube.com/watch?v=GJNjaRJWVP8"

if not Path(DEFAULT_VIDEO).exists():
    print("Downloading test video...")
    ret = !yt-dlp -f 'bestvideo[height<=1080]+bestaudio/best[height<=1080]' \
        --merge-output-format mp4 -o "{DEFAULT_VIDEO}" "{DEFAULT_URL}"

if Path(DEFAULT_VIDEO).exists() and Path(DEFAULT_VIDEO).stat().st_size > 0:
    input_video = DEFAULT_VIDEO
    print(f"Using: {input_video}")
else:
    print("yt-dlp failed. Upload a CCTV video file.")
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        input_video = list(uploaded.keys())[0]
        print(f"Using uploaded: {input_video}")
    else:
        raise FileNotFoundError("No video file provided.")

## Run Analysis

All features enabled: detection, tracking, zones, gates, calibration,
interactions, evidence capture, and **vehicle ANPR**.

In [ ]:
import torch
import json
from src.pipeline import analyze_video

zone_config = json.load(open("configs/demo_zones.json"))
calibration_config = json.load(open("configs/demo_calibration.json"))

result = analyze_video(
    video_path=input_video,
    output_dir="outputs",
    model_family="yolo11",
    model_size="xlarge",
    conf_threshold=0.4,
    device="cuda" if torch.cuda.is_available() else "cpu",
    use_reid=True,
    reid_model="x1_0",
    track_thresh=0.4,
    match_thresh=0.7,
    track_low_thresh=0.1,
    track_buffer=450,
    zone_config=zone_config,
    calibration_config=calibration_config,
    capture_evidence=True,
    filter_stationary_objects=True,
    min_move_distance=20.0,
)
print("\nDone.")

## Summary Report

Includes object counts, gate tallies, events, **vehicle intelligence**, and **scene understanding** (carrying detection, overloaded vehicles).

In [ ]:
from pathlib import Path
summary = Path("outputs/summary.txt")
if summary.exists():
    print(summary.read_text())

# Vehicle summary
import json
analytics = json.load(open("outputs/analytics.json"))
v = analytics.get("vehicles", {})
if v.get("total_vehicles", 0) > 0:
    print(f"\nVEHICLES: {v['total_vehicles']} total")
    print(f"  With plates:  {v['with_plates']}")
    print(f"  Without:      {v['without_plates']}")
    print(f"  Total visits: {v['total_visits']}")
    for vl in analytics.get("vehicle_list", [])[:5]:
        print(f"  {vl.get('plate') or 'unknown':>8s}  {vl['color']:>7s}  {vl['vehicle_type']:>10s}  "
              f"{vl['visit_count']} visit(s)")

# Scene understanding summary
se = analytics.get("scene_events", {})
if se.get("person_carrying", 0) > 0 or se.get("overloaded_vehicle", 0) > 0:
    print(f"\nSCENE EVENTS:")
    print(f"  Person carrying:  {se.get('person_carrying', 0)}")
    print(f"  Overloaded veh:   {se.get('overloaded_vehicle', 0)}")
    for ev in analytics.get("events", []):
        if ev["type"] in ("person_carrying", "overloaded_vehicle"):
            print(f"    [{ev['type']}] {ev['message']}")

## Download Outputs

Annotated video, full analytics JSON, and summary.

In [ ]:
from google.colab import files
files.download("outputs/output_tracking.mp4")
files.download("outputs/analytics.json")
files.download("outputs/summary.txt")

## Production Features

### 1. Config System (edit YAML, no code changes)

In [ ]:
from src.config import ConfigLoader
cfg = ConfigLoader()
print("Detector:", cfg.load("detector").raw)
print("\nTracker:", cfg.load("tracker").raw)
print("\nAnalytics:", cfg.load("analytics").raw)
print("\nVehicle:", cfg.load("vehicle").raw)

### 2. TensorRT (NVIDIA GPU only — 2-5x speedup)

In [ ]:
from src.optimization.tensorrt_export import export_to_engine, has_engine
if torch.cuda.is_available() and not has_engine("yolo11", "xlarge"):
    path = export_to_engine(model_family="yolo11", model_size="xlarge", device=0)
    print(f"TensorRT engine: {path}")
else:
    print("Engine exists or CUDA unavailable, skipping.")

### 3. Multi-Camera (4-feed test)

Process all 4 feeds (original + mirrored/slow/reverse combos) and compare results.

In [ ]:
import sys, json, subprocess
from pathlib import Path
import torch
sys.path.insert(0, '.')

# Generate the 3 transformed videos from the downloaded original
ORIGINAL = "The CCTV People Demo 2.mp4"
TEST_DIR = Path("test_videos")
TEST_DIR.mkdir(exist_ok=True)

FEEDS = {
    "cam00_original": ORIGINAL,
    "cam01_mirror_slow": TEST_DIR / "cam01_mirror_slow.mp4",
    "cam02_mirror_reverse": TEST_DIR / "cam02_mirror_reverse.mp4",
    "cam03_slow_reverse": TEST_DIR / "cam03_slow_reverse.mp4",
}

if not FEEDS["cam01_mirror_slow"].exists():
    print("Generating transformed test videos...")
    def gen(name, filt):
        dst = str(TEST_DIR / name)
        subprocess.run(["ffmpeg", "-y", "-i", ORIGINAL, "-vf", filt,
                       "-an", "-c:v", "libx264", "-preset", "fast", dst],
                       capture_output=True)
        print(f"  Created: {name}")
    gen("cam01_mirror_slow.mp4", "hflip,setpts=2.0*PTS")
    gen("cam02_mirror_reverse.mp4", "hflip,reverse")
    gen("cam03_slow_reverse.mp4", "reverse,setpts=2.0*PTS")

OUTPUT_BASE = Path("outputs") / "4feeds"
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

from src.pipeline import analyze_video

for cam_name, video_path in FEEDS.items():
    print(f"\n=== {cam_name} ===")
    zone_config = json.load(open("configs/demo_zones.json")) if Path("configs/demo_zones.json").exists() else {}
    calib_config = json.load(open("configs/demo_calibration.json")) if Path("configs/demo_calibration.json").exists() else {}
    result = analyze_video(
        video_path=str(video_path),
        output_dir=str(OUTPUT_BASE / cam_name),
        model_family="yolo11",
        model_size="xlarge" if torch.cuda.is_available() else "nano",
        conf_threshold=0.4,
        device="cuda" if torch.cuda.is_available() else "cpu",
        max_frames=200,
        zone_config=zone_config,
        calibration_config=calib_config,
        capture_evidence=True,
        filter_stationary_objects=True,
        min_move_distance=20.0,
    )
    print(f"  Tracks: {result['total_objects_tracked']}, Events: {len(result['events'])}")

### 4. Cross-Camera Comparison

In [ ]:
report = json.load(open(OUTPUT_BASE / "4feeds_report.json"))
print(f"Cameras: {report['camera_count']}\n")
print(f"{'Metric':25s}", end="")
for cn in report['camera_names']:
    print(f"{cn:20s}", end="")
print()
print("-" * (25 + 20 * report['camera_count']))
print(f"{'Tracks':25s}", end="")
for cn in report['camera_names']:
    print(f"{report['per_camera'][cn]['tracks']:20d}", end="")
print()
print(f"{'Events':25s}", end="")
for cn in report['camera_names']:
    print(f"{report['per_camera'][cn]['events']:20d}", end="")
print("\n")
for etype, cam_counts in report['events_by_type'].items():
    print(f"{etype:25s}", end="")
    for cn in report['camera_names']:
        print(f"{cam_counts[cn]:20d}", end="")
    print()

# Show mosaic if available
from IPython.display import Video
mosaic_path = list(Path("outputs/4feeds").glob("mosaic_*.mp4"))
if mosaic_path:
    display(Video(str(mosaic_path[0]), width=960))